# Sims Dorp

Een levenssimulatie van een gewoon dorp met vijf bewoners.

**Bewoners:** Emma, Lars, Fatima, Daan, Sofia
**API:** NewsAPI (sentiment van nieuws beïnvloedt stemming) — komt in stap 3


In [ ]:
import os
import tkinter as tk
from tkinter import simpledialog
from abc import abstractmethod
import random
from dotenv import load_dotenv

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")
if not GOOGLE_API_KEY:
    print("Zet GOOGLE_API_KEY in een .env bestand")
else:
    print("API key geladen")


## Configuratie


In [ ]:
GRID_SIZE = 15
CELL_SIZE = 40
DEFAULT_SPEED = 2000


## SimsWereld basisklasse


In [ ]:
class SimsWereld:
    def __init__(self, root):
        self.root = root
        self.root.title("Sims Dorp")
        self.canvas = tk.Canvas(root, width=GRID_SIZE * CELL_SIZE, height=GRID_SIZE * CELL_SIZE)
        self.canvas.bind("<Button-1>", self.on_canvas_click)
        self.canvas.pack()
        self.speed = DEFAULT_SPEED
        self.running = False
        self.step_mode = False
        self.stopping = False
        self.root.protocol("WM_DELETE_WINDOW", self._on_close)
        self.grid = [[{} for _ in range(GRID_SIZE)] for _ in range(GRID_SIZE)]
        self.initialiseer_wereld()
        self.draw_grid()
        self.control_frame = tk.Frame(root)
        self.control_frame.pack()
        self.create_buttons()
        self.root.after(self.speed, self.update_world)

    def _on_close(self):
        print("Afsluiten...")
        self.stopping = True
        self.root.destroy()

    def draw_grid(self):
        self.canvas.delete("all")
        for y in range(GRID_SIZE):
            for x in range(GRID_SIZE):
                x1, y1 = x * CELL_SIZE, y * CELL_SIZE
                x2, y2 = x1 + CELL_SIZE, y1 + CELL_SIZE
                self.canvas.create_rectangle(x1, y1, x2, y2, fill="white", outline="#ccc")
                cel = self.grid[y][x]
                if cel:
                    self.canvas.create_text(
                        x1 + CELL_SIZE // 2, y1 + CELL_SIZE // 2,
                        text=cel.get("icon", "?"), font=("Arial", 16)
                    )

    def create_buttons(self):
        def play():
            self.step_mode = False
            self.running = not self.running

        def stap():
            self.step_mode = True
            self.running = True

        def snelheid():
            val = simpledialog.askinteger("Snelheid", "Snelheid in ms:", initialvalue=self.speed)
            if val:
                self.speed = val

        def mededeling():
            instructie = simpledialog.askstring("Instructie", "Geef alle bewoners een opdracht:")
            if instructie:
                for y in range(GRID_SIZE):
                    for x in range(GRID_SIZE):
                        if self.grid[y][x].get("type") == "bewoner":
                            self.instrueer(self.grid[y][x], instructie)

        tk.Button(self.control_frame, text="▶️ Play/Pauze", command=play).grid(row=0, column=0, padx=4)
        tk.Button(self.control_frame, text="⏭️ Stap", command=stap).grid(row=0, column=1, padx=4)
        tk.Button(self.control_frame, text="⏩ Snelheid", command=snelheid).grid(row=0, column=2, padx=4)
        tk.Button(self.control_frame, text="📢 Instructie", command=mededeling).grid(row=0, column=3, padx=4)

    def update_world(self):
        if self.running:
            self.doe_alles()
            self.draw_grid()
            if self.step_mode:
                self.running = False
        if not self.stopping:
            self.root.after(self.speed, self.update_world)

    def on_canvas_click(self, event):
        gx = event.x // CELL_SIZE
        gy = event.y // CELL_SIZE
        if 0 <= gx < GRID_SIZE and 0 <= gy < GRID_SIZE:
            cel = self.grid[gy][gx]
            if cel.get("type") == "bewoner":
                opdracht = simpledialog.askstring(
                    "Instructie", f"Geef {cel.get('naam', 'bewoner')} een opdracht:"
                )
                if opdracht:
                    self.instrueer(cel, opdracht)

    @abstractmethod
    def initialiseer_wereld(self):
        pass

    @abstractmethod
    def doe_alles(self):
        pass

    def instrueer(self, sim: dict, instructie: str):
        if instructie:
            sim["instructies"].append(instructie)


## Dorp: objecten en bewoners


In [ ]:
OBJECTEN = {
    "🌳": "boom",
    "🏠": "huis",
    "☕": "cafe",
    "🛒": "supermarkt",
    "📰": "kiosk",
    "🌿": "park",
}

BEWONERS = [
    {"naam": "Emma",   "icon": "😊", "persoonlijkheid": "optimistisch en sociaal"},
    {"naam": "Lars",   "icon": "😏", "persoonlijkheid": "cynisch maar grappig"},
    {"naam": "Fatima", "icon": "🌸", "persoonlijkheid": "zorgzaam en empathisch"},
    {"naam": "Daan",   "icon": "😐", "persoonlijkheid": "nuchter en praktisch"},
    {"naam": "Sofia",  "icon": "🎨", "persoonlijkheid": "creatief en gevoelig"},
]

VASTE_OBJECTEN = [
    # Bomen in de hoeken
    (0, 0, "🌳"), (14, 0, "🌳"), (0, 14, "🌳"), (14, 14, "🌳"),
    (1, 0, "🌳"), (0, 1, "🌳"), (13, 0, "🌳"), (14, 1, "🌳"),
    # Park in het midden
    (6, 6, "🌿"), (7, 6, "🌿"), (8, 6, "🌿"),
    (6, 7, "🌿"), (7, 7, "🌿"), (8, 7, "🌿"),
    (6, 8, "🌿"), (7, 8, "🌿"), (8, 8, "🌿"),
    # Café
    (3, 3, "☕"),
    # Supermarkt
    (11, 3, "🛒"),
    # Kiosk
    (7, 2, "📰"),
    # Huizen
    (2, 11, "🏠"), (5, 12, "🏠"), (9, 12, "🏠"), (12, 11, "🏠"), (7, 11, "🏠"),
]

STARTPOSITIES = [(3, 7), (7, 4), (11, 7), (5, 5), (10, 10)]


## Stap 2: LLM-call

Een simpele chain die op basis van de situatie van de Sim één actie kiest:
`beweeg`, `eet`, `rust`, of `praat`.

Elke 3 ticks vraagt een Sim het aan de LLM. Tussendoor herhaalt hij de vorige actie.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", google_api_key=GOOGLE_API_KEY, temperature=0.7)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Je bent {naam}, een bewoner van een klein dorp. "
     "Jouw persoonlijkheid: {persoonlijkheid}. "
     "Gedraag je zoals jouw persoonlijkheid dat zou doen. "
     "Gebruik altijd vriendelijke, kindvriendelijke taal."),
    ("human",
     "Jouw situatie:\n"
     "- Honger: {honger}/10\n"
     "- Stemming: {stemming}\n"
     "- Objecten in de buurt: {objecten_nabij}\n"
     "- Instructie van speler: {instructie}\n\n"
     "Kies precies één actie: beweeg, eet, rust, of praat. "
     "Geef alleen dat ene woord terug, verder niets.")
])

kies_actie = prompt | llm | StrOutputParser()

# Test of de chain werkt
test = kies_actie.invoke({
    "naam": "Emma",
    "persoonlijkheid": "optimistisch en sociaal",
    "honger": 3,
    "stemming": "hongerig",
    "objecten_nabij": "supermarkt, park",
    "instructie": "geen",
})
print("Test LLM-output:", test)


## DorpWereld implementatie


In [ ]:
class DorpWereld(SimsWereld):
    def __init__(self):
        self.tick = 0
        self.llm_interval = 3  # elke 3 ticks een LLM-call per Sim
        super().__init__(tk.Tk())
        self.root.mainloop()

    def initialiseer_wereld(self):
        for x, y, icon in VASTE_OBJECTEN:
            self.grid[y][x] = {"type": OBJECTEN[icon], "icon": icon}

        for idx, (bewoner, (sx, sy)) in enumerate(zip(BEWONERS, STARTPOSITIES)):
            self.grid[sy][sx] = {
                "type": "bewoner",
                "icon": bewoner["icon"],
                "naam": bewoner["naam"],
                "persoonlijkheid": bewoner["persoonlijkheid"],
                "honger": 8,
                "stemming": "neutraal",
                "instructies": [],
                "idx": idx,
                "actie": "beweeg",  # laatste LLM-beslissing
            }
            print(f"{bewoner['naam']} staat op ({sx}, {sy})")

    def _objecten_nabij(self, x, y, straal=4):
        gevonden = set()
        for dy in range(-straal, straal + 1):
            for dx in range(-straal, straal + 1):
                ny, nx = y + dy, x + dx
                if 0 <= ny < GRID_SIZE and 0 <= nx < GRID_SIZE:
                    cel = self.grid[ny][nx]
                    if cel and cel.get("type") != "bewoner":
                        gevonden.add(cel["type"])
        return list(gevonden) if gevonden else ["geen"]

    def doe_alles(self):
        self.tick += 1

        posities = [
            (y, x)
            for y in range(GRID_SIZE)
            for x in range(GRID_SIZE)
            if self.grid[y][x].get("type") == "bewoner"
        ]

        for y, x in posities:
            if self.grid[y][x].get("type") != "bewoner":
                continue
            sim = self.grid[y][x]

            sim["honger"] = max(0, sim["honger"] - 1)

            # LLM-call elke llm_interval ticks (offset per Sim zodat ze niet tegelijk bellen)
            if (self.tick + sim["idx"]) % self.llm_interval == 0:
                instructie = sim["instructies"].pop(0) if sim["instructies"] else "geen"
                try:
                    actie = kies_actie.invoke({
                        "naam": sim["naam"],
                        "persoonlijkheid": sim["persoonlijkheid"],
                        "honger": sim["honger"],
                        "stemming": sim["stemming"],
                        "objecten_nabij": ", ".join(self._objecten_nabij(x, y)),
                        "instructie": instructie,
                    }).strip().lower()
                    # Zorg dat de output een geldige actie is
                    if actie not in ("beweeg", "eet", "rust", "praat"):
                        actie = "beweeg"
                    sim["actie"] = actie
                    print(f"[{sim['naam']}] kiest: {actie}")
                except Exception as e:
                    print(f"[{sim['naam']}] LLM-fout: {e}")

            # Actie uitvoeren
            actie = sim.get("actie", "beweeg")

            if actie == "rust":
                pass  # Niet bewegen

            elif actie == "praat":
                for dy2, dx2 in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                    buur = self.grid[y + dy2][x + dx2] if 0 <= y + dy2 < GRID_SIZE and 0 <= x + dx2 < GRID_SIZE else {}
                    if buur.get("type") == "bewoner":
                        print(f"[{sim['naam']}] praat met {buur['naam']}")
                        break

            else:  # beweeg of eet
                dx, dy = random.choice([(0, 1), (1, 0), (0, -1), (-1, 0)])
                new_x = max(0, min(GRID_SIZE - 1, x + dx))
                new_y = max(0, min(GRID_SIZE - 1, y + dy))
                doel = self.grid[new_y][new_x]
                doel_type = doel.get("type") if doel else None

                if not doel:
                    self.grid[new_y][new_x] = sim
                    self.grid[y][x] = {}
                    x, y = new_x, new_y
                elif doel_type == "supermarkt" and sim["honger"] < 6:
                    sim["honger"] = min(10, sim["honger"] + 5)
                    print(f"[{sim['naam']}] koopt eten (honger → {sim['honger']})")
                elif doel_type == "cafe" and sim["honger"] < 8:
                    sim["honger"] = min(10, sim["honger"] + 2)
                    print(f"[{sim['naam']}] drinkt koffie (honger → {sim['honger']})")

            # Stemming bijwerken
            if sim["honger"] > 7:
                sim["stemming"] = "blij"
            elif sim["honger"] > 4:
                sim["stemming"] = "neutraal"
            else:
                sim["stemming"] = "hongerig"


## Start het spel


In [ ]:
DorpWereld()
